# Setup

In [0]:
%pip install nhp_data

In [0]:
from pyspark.sql import functions as F
from nhp.data.get_spark import get_spark
from nhp.data.table_names import table_names
from nhp.data.inputs_data.ip.rates import get_population
from nhp.data.inputs_data.direct_standardisation import directly_standardise
from nhp.data.inputs_data.helpers import complete_age_sex_rows
from nhp.data.raw_data.mitigators.ip.hes_datasets import (
    any_diagnosis,
    #any_procedure,
    nhp_apc,
    primary_diagnosis,
    #primary_procedure,
    #secondary_diagnosis,
)
from pyspark.sql.window import Window

In [0]:
# Lakemart where we're saving outputs
lakeNameSU = "udalstdataanalysisprod.dfs.core.windows.net"
containerNameSU = "analytics-projects"
tpma_review_path = "abfss://"+containerNameSU+"@"+lakeNameSU+"/StrategyUnit/StrategyUnit/analytics_modelling/nhp_tpma_review/2026_09_discharge_ready_date/"

# HES completeness over time

`disreadydate` not well populated until 23/24:

In [0]:
spark.read.table(table_names.hes_apc).limit(10).display()

In [0]:
hes_discharge_ready_date = (
    spark.read.table(table_names.hes_apc)
    .withColumn("Der_Activity_Month", F.regexp_replace(F.substring(F.col("disdate"), 1, 7), "-", ""))
    .filter(F.col("last_episode_in_spell") & ~F.col("disdate").isNull() & (F.col("Der_Activity_Month") >= 202309))
    # exclude under 16s, applying the age logic from https://github.com/The-Strategy-Unit/nhp_data/blob/main/src/nhp/data/nhp_datasets/apc.py:
    .withColumn(
        "age",
        F.when(
            (F.col("admiage") == 999) | F.col("admiage").isNull(),
            F.when(F.col("startage") > 7000, 0).otherwise(F.col("startage")),
        ).otherwise(F.col("admiage")),
    )
    .filter(F.col("age") >= 16)
    # exclude zero length of stay:
    .filter(F.datediff(F.col("disdate"), F.col("admidate")) > 0)
    # exclude method of admission other than elective or emergency:
    .filter(~F.col("admimeth").isin("31", "32", "82", "83"))
    # exclude those who die in hospital and those transferred to other hospitals/hospices:
    .filter(~F.col("disdest").isin("51", "52", "87", "88", "79"))
    .filter(~F.col("dismeth").isin(4, 5))    
    # filter to "specific acute" treatment function groups (see appendix for more info):
    .filter(F.col("tretspef").isin(
        '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '113', '115', '120', '130', '140', '141', '142', '143', '144', '145', '150', '160', '161', '170', '171', '172', '173', '174', '180', '190', '191', '192', '200', '211', '212', '213', '214', '215', '216', '217', '218', '219', '220', '221', '222', '230', '240', '241', '242', '250', '251', '252', '253', '254', '255', '256', '257', '258', '259', '261', '262', '263', '264', '270', '280', '300', '301', '302', '303', '304', '305', '306', '307', '308', '309', '310', '311', '313', '314', '315', '316', '317', '318', '319', '320', '321', '322', '323', '324', '325', '326', '327', '328', '329', '330', '333', '340', '341', '342', '343', '347', '348', '350', '352', '361', '370', '371', '400', '401', '410', '420', '421', '422', '430', '431', '450', '451', '460', '461', '502', '503', '505', '663', '670', '673', '675', '677', '800', '811', '812', '822', '834'
        ))
    .withColumn("has_disreadydate", F.when(F.col("disreadydate").isNull(), F.lit(0)).otherwise(F.lit(1)))
)

In [0]:
hes_discharge_ready_date_complete = (
    hes_discharge_ready_date
    .groupBy(F.col("Der_Activity_Month"))
    .agg(
        F.sum(F.col("has_disreadydate")).alias("has_disreadydate"),
        F.count(F.col("Der_Activity_Month")).alias("number_spells")
        )
)

hes_discharge_ready_date_complete.display()

# SUS completeness over time

In [0]:
spark.read.table("udal_gold_restricted.mesh_apc.apcs_core_monthly_snapshot").limit(10).display()

In [0]:
sus_discharge_ready_date_complete = (
    spark.read.table("udal_gold_restricted.mesh_apc.apcs_core_monthly_snapshot")
    .filter((~F.col("Discharge_Date").isNull()) & (F.col("Der_Activity_Month") >= 202309))
    # exclude under 16s, applying the age logic from https://github.com/The-Strategy-Unit/nhp_data/blob/main/src/nhp/data/nhp_datasets/apc.py:
    .withColumn(
        "age",
        F.when(
            (F.col("Der_Age_At_CDS_Activity_Date") == 999) | F.col("Der_Age_At_CDS_Activity_Date").isNull(),
            F.when(F.col("Age_At_Start_of_Spell_SUS") > 7000, 0).otherwise(F.col("Age_At_Start_of_Spell_SUS")),
        ).otherwise(F.col("Der_Age_At_CDS_Activity_Date")),
    )    
    .filter(F.col("age") >= 16)
    # exclude zero length of stay:
    .filter(F.datediff(F.col("Discharge_Date"), F.col("Admission_Date")) > 0)
    # exclude method of admission other than elective or emergency:
    .filter(~F.col("Admission_Method").isin("31", "32", "82", "83"))
    # exclude those who die in hospital and those transferred to other hospitals/hospices:
    .filter(~F.col("Discharge_Destination").isin("51", "52", "87", "88", "79"))
    .filter(~F.col("Discharge_Method").isin(4, 5))
    # filter to "specific acute" treatment function groups (see appendix for more info):
    .filter(F.col("Der_Dischg_Treatment_Function_Code").isin(
        '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '113', '115', '120', '130', '140', '141', '142', '143', '144', '145', '150', '160', '161', '170', '171', '172', '173', '174', '180', '190', '191', '192', '200', '211', '212', '213', '214', '215', '216', '217', '218', '219', '220', '221', '222', '230', '240', '241', '242', '250', '251', '252', '253', '254', '255', '256', '257', '258', '259', '261', '262', '263', '264', '270', '280', '300', '301', '302', '303', '304', '305', '306', '307', '308', '309', '310', '311', '313', '314', '315', '316', '317', '318', '319', '320', '321', '322', '323', '324', '325', '326', '327', '328', '329', '330', '333', '340', '341', '342', '343', '347', '348', '350', '352', '361', '370', '371', '400', '401', '410', '420', '421', '422', '430', '431', '450', '451', '460', '461', '502', '503', '505', '663', '670', '673', '675', '677', '800', '811', '812', '822', '834'
        ))
    .withColumn("has_disreadydate", F.when(F.col("Discharge_Ready_Date").isNull(), F.lit(0)).otherwise(F.lit(1)))
    .groupBy(F.col("Der_Activity_Month"))
    .agg(
        F.sum(F.col("has_disreadydate")).alias("has_disreadydate"),
        F.count(F.col("Der_Activity_Month")).alias("number_spells")
        )
)

sus_discharge_ready_date_complete.display()

# Saving completeness data

In [0]:
discharge_ready_date_complete = (
    hes_discharge_ready_date_complete
    .withColumn("source", F.lit("hes"))
    .union(
        sus_discharge_ready_date_complete
        .withColumn("source", F.lit("sus"))
    )
)

# Write parquet file
discharge_ready_date_complete.write.mode("overwrite").parquet(tpma_review_path + "discharge_ready_date_complete.parquet")

# HES negative delay duration

In [0]:
hes_discharge_ready_date_negative = (
    hes_discharge_ready_date
    .filter(F.col("fyear") == 202425)
    .withColumn(
        "disreadydate_valid", 
        F.when(F.col("disreadydate").isNull(), F.col("disdate"))
        .otherwise(F.col("disreadydate"))
        )    
    .withColumn("delay_duration", F.datediff(F.col("disdate"), F.col("disreadydate_valid"))) 
    .filter(F.col("delay_duration") < 0)
    .groupBy("delay_duration")
    .agg(
        F.count("fyear").alias("number_spells")
        )
)

# Write parquet file
hes_discharge_ready_date_negative.write.mode("overwrite").parquet(tpma_review_path + "hes_discharge_ready_date_negative.parquet")

# HES early discharge ready dates

In [0]:
hes_early_discharge_ready_dates = (
    hes_discharge_ready_date
    .filter(F.col("fyear") == 202425)
    .withColumn(
            "disreadydate_valid", 
            F.when(F.col("disreadydate").isNull(), F.col("disdate"))
            .when(F.col("disreadydate") > F.col("disdate"), F.col("disdate"))
            .otherwise(F.col("disreadydate"))
            )   
    .filter(F.col("disreadydate") == "1900-01-01") 
    .groupBy("der_activity_month")
    .agg(F.count("fyear").alias("number_spells"))
)

# Write parquet file
hes_early_discharge_ready_dates.write.mode("overwrite").parquet(tpma_review_path + "hes_early_discharge_ready_dates.parquet")

# HES outliers

In [0]:
hes_discharge_ready_date_outliers = (
    hes_discharge_ready_date
    .filter(F.col("fyear") == 202425)
    .withColumn(
            "disreadydate_valid", 
            F.when(F.col("disreadydate").isNull(), F.col("disdate"))
            .when(F.col("disreadydate") > F.col("disdate"), F.col("disdate"))
            .when(F.col("disreadydate") == "1900-01-01", F.col("disdate"))
            .otherwise(F.col("disreadydate"))
            )   
    .withColumn("delay_duration", F.datediff(F.col("disdate"), F.col("disreadydate_valid"))) 
    .groupBy("der_activity_month", "delay_duration")
    .agg(
        F.count("fyear").alias("number_spells")
        )
)

# Write parquet file
hes_discharge_ready_date_outliers.write.mode("overwrite").parquet(tpma_review_path + "hes_discharge_ready_date_outliers.parquet")

# Appendix



## "Specific acute" treatment function codes

Better Care Fund says to restrict spells to those with "specific acute" treatment function codes. 

A table of "specific acute" treatment function codes was found at https://www.england.nhs.uk/london/wp-content/uploads/sites/2/2023/12/Specific_Acute_TFC.pdf. Copying and pasting this into the following R code:


In [0]:
%skip
copy_paste <- ("PASTE COPIED CONTENT HERE")

raw <- copy_paste |>
  stringr::str_split(" ") 

split <- raw[[1]] |>
  as.numeric()

paste(split, collapse = "', '") |>
  stringr::str_remove_all(", 'NA'") |>
  stringr::str_remove(", 'NA")

gives the "specific acute" treatment function codes:


In [0]:
%skip
'100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '113', '115', '120', '130', '140', '141', '142', '143', '144', '145', '150', '160', '161', '170', '171', '172', '173', '174', '180', '190', '191', '192', '200', '211', '212', '213', '214', '215', '216', '217', '218', '219', '220', '221', '222', '230', '240', '241', '242', '250', '251', '252', '253', '254', '255', '256', '257', '258', '259', '261', '262', '263', '264', '270', '280', '300', '301', '302', '303', '304', '305', '306', '307', '308', '309', '310', '311', '313', '314', '315', '316', '317', '318', '319', '320', '321', '322', '323', '324', '325', '326', '327', '328', '329', '330', '333', '340', '341', '342', '343', '347', '348', '350', '352', '361', '370', '371', '400', '401', '410', '420', '421', '422', '430', '431', '450', '451', '460', '461', '502', '503', '505', '663', '670', '673', '675', '677', '800', '811', '812', '822', '834'

## Initial exploration

Some code I used initially to explore DRD field, but have not used in final outputs.

### 24/25 completeness

In 24/25, 24% of spells had a discharge ready date:

In [0]:
display(
    spark.read.table(table_names.hes_apc)
    .filter((F.col("fyear") == 202425) & F.col("last_episode_in_spell") & ~F.col("disdate").isNull())
    .withColumn("has_disreadydate", F.when(F.col("disreadydate").isNull(), F.lit(0)).otherwise(F.lit(1)))
    .agg(F.sum(F.col("has_disreadydate")).alias("has_disreadydate"),
         F.count(F.col("fyear")).alias("number_spells"))
    .withColumn("perc", F.col("has_disreadydate") * 100 / F.col("number_spells"))
)

### Providers 

Data from 181 providers, so probably not an issue with providers not submitting:

In [0]:
display(
    spark.read.table(table_names.hes_apc)
    .filter((F.col("fyear") == 202425) & F.col("last_episode_in_spell") & ~F.col("disdate").isNull())
    .filter(~F.col("disreadydate").isNull())
    .select(F.col("procode3"))
    .distinct()
    .count()
)

### Discharge ready date = Discharge date

How many spells have a discharge ready date that equals the discharge date.

In [0]:
# HES:
display(
    spark.read.table(table_names.hes_apc)
    .filter((F.col("fyear") == 202425) & F.col("last_episode_in_spell") & ~F.col("disdate").isNull())
    .filter(~F.col("disreadydate").isNull())
    .withColumn("dates_equal", F.when(F.col("disreadydate") == F.col("disdate"), F.lit(1)).otherwise(F.lit(0)))    
    .agg(F.sum(F.col("dates_equal")).alias("dates_equal"),
         F.count(F.col("fyear")).alias("number_disreadydate_spells"))
    .withColumn("perc", F.col("dates_equal") * 100 / F.col("number_disreadydate_spells"))
)

In [0]:
# SUS:
display(
    spark.read.table("udal_gold_restricted.mesh_apc.apcs_core_monthly_snapshot")
    .withColumn("fyear", F.regexp_replace("Der_Financial_Year", "/", ""))
    .filter((F.col("fyear") == 202425) & ~F.col("Discharge_Date").isNull())
    .filter(~F.col("Discharge_Ready_Date").isNull())
    .withColumn("dates_equal", F.when(F.col("Discharge_Ready_Date") == F.col("Discharge_Date"), F.lit(1)).otherwise(F.lit(0)))   
    .agg(F.sum(F.col("dates_equal")).alias("dates_equal"),
         F.count(F.col("fyear")).alias("number_disreadydate_spells"))
    .withColumn("perc", F.col("dates_equal") * 100 / F.col("number_disreadydate_spells"))
)

### Critical care

Wanted to check if the spells with a discharge ready date were all critical care related.

There were some columns in HES APC for critical care, but moved to new dataset HES critical care.

The HES APC data comes from `udal_silver_restricted.seft_hes.hesapc`. There is `udal_silver_restricted.seft_hes.hescc` for critical care data, but I can't access `udal_silver_restricted.seft_hes.*` to look at it.

So couldn't check.